In [ ]:
from tqdm import tqdm
from datasets import load_dataset


openreason_data = load_dataset("nvidia/OpenCodeReasoning-2", trust_remote_code=True, split="python")

In [ ]:
#and float(data["pass_rate"]) <= 0.72
def filter_data(data):
    r1_generation = data["r1_generation"]
    if float(data["pass_rate"]) >= 0.45 and len(r1_generation) < 12500 and data["judgement"] == "right" and "python" in r1_generation:
        return True
    return False

filtered_data = openreason_data.filter(filter_data)

In [ ]:
filtered_data = filtered_data.map(lambda x: {"pass_rate": float(x["pass_rate"])})

filtered_data = filtered_data.sort("pass_rate", reverse=False)

In [ ]:
len(filtered_data)

In [ ]:
test_queries = []
import json

for ds_name in ["code_contests" , "codeforces", "humanevalplus", "leetcode"]:
    path = f"/fsx/home/skokane/research/verl_mt/rllm/data/test/code/{ds_name}.json"
    with open(path, "r") as f:
        data = json.load(f)
    for item in data:
        test_queries.append(item["problem"])
        
len(test_queries)

In [ ]:
hf_datasets = {
    "taco": load_dataset("BAAI/TACO", trust_remote_code=True),
    "apps": load_dataset("codeparrot/apps", trust_remote_code=True),
    "code_contests": load_dataset("deepmind/code_contests"),
    "open-r1/codeforces": load_dataset("open-r1/codeforces")
}

dt = {"taco": "input_output", "apps": "input_output", "code_contests": "generated_tests", "open-r1/codeforces": "official_tests"}


def get_question(ds_name, split, index):
    benchmark = hf_datasets[ds_name][split][int(index)]
    if ds_name == "code_contests":
        if not benchmark["description"]:
            return None, None
        return benchmark["description"], {"inputs": benchmark[dt[ds_name]]["input"], "outputs": benchmark[dt[ds_name]]["output"]}
    elif ds_name in ["taco", "apps"]:
        return benchmark["question"], json.loads(benchmark[dt[ds_name]])
    elif ds_name == "open-r1/codeforces":
        if not benchmark["description"]:
            return None, None
        question = benchmark["description"]
        if benchmark["input_format"]:
            question += "\n\nInput\n\n" + benchmark["input_format"]
        if benchmark["output_format"]:
            question += "\n\nOutput\n\n" + benchmark["output_format"]
        if benchmark["examples"]:
            question += "\n\nExamples"
            for example in benchmark["examples"]:
                if "input" in example:
                    question += "\n\nInput\n\n" + example["input"]
                if "output" in example:
                    question += "\n\nOutput\n\n" + example["output"]
        if benchmark["note"]:
            question += "\n\nNote\n\n" + benchmark["note"]
        return question , benchmark[dt[ds_name]]

    return None, None


filter_data = {"taco": [], "apps": [], "code_contests": [], "open-r1/codeforces": []}
n = len(filtered_data)
count = 0
for idx in tqdm(range(n)):
    ocr2_ds_item = filtered_data[idx]
    ocr2_ds_item["tests"] = ""
    assert ocr2_ds_item["dataset"] in ["taco", "apps", "code_contests", "open-r1/codeforces"]
    ds_name, ds_split, ds_index = ocr2_ds_item["dataset"], ocr2_ds_item["split"], int(ocr2_ds_item["index"])
    question, tests = get_question(ds_name, ds_split, ds_index)
    if question is not None and tests is not None and len(question) <= 2048: 
        # assert ocr2_ds_item["question"] == "-"
        ocr2_ds_item["question"] = question
        ocr2_ds_item["tests"] = tests
        filter_data[ocr2_ds_item["dataset"]].append(ocr2_ds_item)
        count += 1
        if count > 12500:
            break

In [ ]:
filter_2_data = {}

for dataname, data in filter_data.items():
    filter_2_data[dataname] = []
    for item in data:
        if item["question"] in test_queries:
            continue
        else:
            filter_2_data[dataname].append(item)

len(filter_2_data)

In [ ]:
for dataname, data in filter_2_data.items():
    if dataname == "open-r1/codeforces":
        dataname = "codeforces"
    with open(f"/fsx/home/skokane/research/verl_mt/rllm/data/train/code/{dataname}.json", "w") as f:
        json.dump(data, f)